# Phase 2 — Measure
## 03 — Sales Consolidation

### Objective
Validate that `Master_data` is the consolidated sales dataset representing:

- `Nov_file`
- `Dec_file`
- `Jan_file`

and then build the governed `fact_sales` table from `Master_data` without double-counting monthly records.

### Key Rules
- Do not append monthly sheets onto `Master_data`.
- Use monthly sheets for reconciliation and validation.
- Preserve repeated Stock Codes until sales grain is understood.
- Map every governed sales row to the canonical `Product_ID`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_path = Path.cwd()

if current_path.name == "Phase_2_Measure":
    PROJECT_ROOT = current_path.parent.parent
elif current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root :", PROJECT_ROOT)
print("Raw data dir :", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

Project root : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System
Raw data dir : d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\raw
Processed dir: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed


In [2]:
## Locate and load sales workbook
sales_files = list(
    RAW_DIR.glob("*Master*Sales*.xlsx")
)

if len(sales_files) != 1:
    raise ValueError(
        f"Expected exactly 1 Master Sales workbook, found {len(sales_files)}"
    )

SALES_FILE = sales_files[0]

sales_excel = pd.ExcelFile(SALES_FILE)

print("Workbook:", SALES_FILE.name)
print("Sheets  :", sales_excel.sheet_names)

Workbook: Master_Sales_data.xlsx
Sheets  : ['Master_data', 'Nov_file', 'Dec_file', 'Jan_file']


In [3]:
master_data = pd.read_excel(
    SALES_FILE,
    sheet_name="Master_data"
)

nov_data = pd.read_excel(
    SALES_FILE,
    sheet_name="Nov_file"
)

dec_data = pd.read_excel(
    SALES_FILE,
    sheet_name="Dec_file"
)

jan_data = pd.read_excel(
    SALES_FILE,
    sheet_name="Jan_file"
)

print("Master_data:", master_data.shape)
print("Nov_file   :", nov_data.shape)
print("Dec_file   :", dec_data.shape)
print("Jan_file   :", jan_data.shape)

Master_data: (966, 11)
Nov_file   : (381, 11)
Dec_file   : (323, 11)
Jan_file   : (262, 11)


In [4]:
## Schema Validation
sheet_columns = pd.DataFrame({
    "Master_data": pd.Series(master_data.columns),
    "Nov_file": pd.Series(nov_data.columns),
    "Dec_file": pd.Series(dec_data.columns),
    "Jan_file": pd.Series(jan_data.columns)
})

display(sheet_columns)

,Master_data,Nov_file,Dec_file,Jan_file
0,Category,Category,Category,Category
1,Stock Code,Stock Code,Stock Code,Stock Code
2,Description,Description,Description,Description
3,Level,Level,Level,Level
4,Sold Period,Sold Period,Sold Period,Sold Period
5,Unit Cost,Unit Cost,Unit Cost,Unit Cost
6,Unit Price,Unit Price,Unit Price,Unit Price
7,Cost Sales,Cost Sales,Cost Sales,Cost Sales
8,Sales Value,Sales Value,Sales Value,Sales Value
9,Profit,Profit,Profit,Profit


In [5]:
sheet_columns = pd.DataFrame({
    "Master_data": pd.Series(master_data.columns),
    "Nov_file": pd.Series(nov_data.columns),
    "Dec_file": pd.Series(dec_data.columns),
    "Jan_file": pd.Series(jan_data.columns)
})

display(sheet_columns)

,Master_data,Nov_file,Dec_file,Jan_file
0,Category,Category,Category,Category
1,Stock Code,Stock Code,Stock Code,Stock Code
2,Description,Description,Description,Description
3,Level,Level,Level,Level
4,Sold Period,Sold Period,Sold Period,Sold Period
5,Unit Cost,Unit Cost,Unit Cost,Unit Cost
6,Unit Price,Unit Price,Unit Price,Unit Price
7,Cost Sales,Cost Sales,Cost Sales,Cost Sales
8,Sales Value,Sales Value,Sales Value,Sales Value
9,Profit,Profit,Profit,Profit


In [6]:
master_cols = list(master_data.columns)

schema_checks = {
    "Nov matches Master": list(nov_data.columns) == master_cols,
    "Dec matches Master": list(dec_data.columns) == master_cols,
    "Jan matches Master": list(jan_data.columns) == master_cols
}

for check, result in schema_checks.items():
    print(f"{check}: {result}")

Nov matches Master: True
Dec matches Master: True
Jan matches Master: True


### Step 1 — First Master vs Monthly Reconciliation

In [7]:
## First Master vs Monthly Reconciliation

monthly_total_rows = (
    len(nov_data)
    + len(dec_data)
    + len(jan_data)
)

print("Master_data rows       :", len(master_data))
print("Nov + Dec + Jan rows   :", monthly_total_rows)
print("Row-count difference   :", len(master_data) - monthly_total_rows)

Master_data rows       : 966
Nov + Dec + Jan rows   : 966
Row-count difference   : 0


In [8]:
## Compare key business totals
metrics = [
    "Sold Period",
    "Cost Sales",
    "Sales Value",
    "Profit"
]

comparison_rows = []

for metric in metrics:
    master_total = master_data[metric].sum()
    
    monthly_total = (
        nov_data[metric].sum()
        + dec_data[metric].sum()
        + jan_data[metric].sum()
    )
    
    comparison_rows.append({
        "Metric": metric,
        "Master_Total": master_total,
        "Monthly_Total": monthly_total,
        "Difference": master_total - monthly_total
    })

master_monthly_reconciliation = pd.DataFrame(
    comparison_rows
)

display(master_monthly_reconciliation)

,Metric,Master_Total,Monthly_Total,Difference
0,Sold Period,1178.00,1178.00,0.000000e+00
1,Cost Sales,273671.79,273671.79,0.000000e+00
2,Sales Value,279350.84,279350.84,-5.820766e-11
3,Profit,5679.05,5679.05,-9.094947e-13


In [9]:
print(
    "Master unique Stock Codes:",
    master_data["Stock Code"].nunique()
)

print(
    "Nov unique Stock Codes:",
    nov_data["Stock Code"].nunique()
)

print(
    "Dec unique Stock Codes:",
    dec_data["Stock Code"].nunique()
)

print(
    "Jan unique Stock Codes:",
    jan_data["Stock Code"].nunique()
)

monthly_unique_union = pd.concat(
    [
        nov_data[["Stock Code"]],
        dec_data[["Stock Code"]],
        jan_data[["Stock Code"]]
    ],
    ignore_index=True
)["Stock Code"].nunique()

print(
    "\nUnique Stock Codes across monthly union:",
    monthly_unique_union
)

Master unique Stock Codes: 768
Nov unique Stock Codes: 381
Dec unique Stock Codes: 319
Jan unique Stock Codes: 262

Unique Stock Codes across monthly union: 768


### Step 2 — Establish Sales Grain

In [10]:
## Add source month to monthly datasets
nov_check = nov_data.copy()
dec_check = dec_data.copy()
jan_check = jan_data.copy()

nov_check["Source_Month"] = "Nov"
dec_check["Source_Month"] = "Dec"
jan_check["Source_Month"] = "Jan"

monthly_union = pd.concat(
    [
        nov_check,
        dec_check,
        jan_check
    ],
    ignore_index=True
)

print("Monthly union shape:", monthly_union.shape)

display(
    monthly_union[
        ["Stock Code", "Description", "Source_Month"]
    ].head()
)

Monthly union shape: (966, 12)


,Stock Code,Description,Source_Month
0,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,Nov
1,112.204,TV Arial Lead 4.0m,Nov
2,DLSC500,Delonghi Descaler,Nov
3,AF01,VACUUM FRESHENERS AF101,Nov
4,SES007NEU0,Sage Descaler (pack of 4),Nov


In [11]:
## How many months does each Stock Code occur in?
product_month_presence = (
    monthly_union
    .groupby("Stock Code")["Source_Month"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("Number_of_Months")
    .to_frame("Product_Count")
)

display(product_month_presence)

,Product_Count
Number_of_Months,
1,602
2,138
3,28


In [12]:
## Validate Stock Code + Month grain
duplicate_product_month = (
    monthly_union
    .duplicated(
        subset=["Stock Code", "Source_Month"]
    )
    .sum()
)

print(
    "Duplicate Stock Code + Month records:",
    duplicate_product_month
)

Duplicate Stock Code + Month records: 4


In [13]:
## Inspect products occurring across all 3 months

three_month_products = (
    monthly_union
    .groupby("Stock Code")["Source_Month"]
    .nunique()
)

three_month_products = three_month_products[
    three_month_products == 3
].index

three_month_sample = (
    monthly_union[
        monthly_union["Stock Code"].isin(
            three_month_products
        )
    ]
    .sort_values(
        ["Stock Code", "Source_Month"]
    )
)

print(
    "Products appearing in all 3 months:",
    len(three_month_products)
)

display(
    three_month_sample[
        [
            "Stock Code",
            "Description",
            "Source_Month",
            "Sold Period",
            "Sales Value",
            "Profit"
        ]
    ].head(30)
)

Products appearing in all 3 months: 28


,Stock Code,Description,Source_Month,Sold Period,Sales Value,Profit
618,12377940,Miele Duoflex HX1 Cat&Dog,Dec,1,207.50,-26.50
889,12377940,Miele Duoflex HX1 Cat&Dog,Jan,1,287.49,53.49
257,12377940,Miele Duoflex HX1 Cat&Dog,Nov,1,290.83,56.83
455,15199,Russell Hobbs 2 Plate Portable Hob,Dec,1,24.99,-5.00
757,15199,Russell Hobbs 2 Plate Portable Hob,Jan,2,49.98,-10.00
79,15199,Russell Hobbs 2 Plate Portable Hob,Nov,1,24.99,-5.00
388,430009,MERCURY 4 GANG SURGE 2M,Dec,2,21.66,9.22
712,430009,MERCURY 4 GANG SURGE 2M,Jan,0,0.01,0.01
9,430009,MERCURY 4 GANG SURGE 2M,Nov,1,9.17,2.95
649,43LQ60006LA.LG,"43"" Smart TV",Dec,2,303.34,-70.66


In [14]:
## Prove Master Rows Correspond to Monthly Rows
business_columns = list(master_data.columns)

master_sorted = (
    master_data[business_columns]
    .sort_values(business_columns)
    .reset_index(drop=True)
)

monthly_sorted = (
    monthly_union[business_columns]
    .sort_values(business_columns)
    .reset_index(drop=True)
)

exact_row_match = master_sorted.equals(
    monthly_sorted
)

print(
    "Master_data exactly equals monthly union:",
    exact_row_match
)

Master_data exactly equals monthly union: True


### Step 3 — Investigate the 4 Product-Month duplicates

In [15]:
duplicate_product_month_rows = (
    monthly_union[
        monthly_union.duplicated(
            subset=["Stock Code", "Source_Month"],
            keep=False
        )
    ]
    .sort_values(
        ["Stock Code", "Source_Month"]
    )
)

print(
    "Rows involved in duplicated Stock Code + Month keys:",
    len(duplicate_product_month_rows)
)

print(
    "Distinct duplicated Stock Code + Month keys:",
    duplicate_product_month_rows[
        ["Stock Code", "Source_Month"]
    ].drop_duplicates().shape[0]
)

display(duplicate_product_month_rows)

Rows involved in duplicated Stock Code + Month keys: 7
Distinct duplicated Stock Code + Month keys: 3


,Category,Stock Code,Description,Level,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Source_Month
521,INSTALLATION,INSTALLATIO,INSTALLATION FEE,-53,12,0.01,125.49,0.12,595.84,595.72,99.98,Dec
522,INSTALLATION,INSTALLATIO,INSTALLATION FREE STANDING,-3,1,0.01,34.49,0.01,33.33,33.32,99.97,Dec
523,INSTALLATION,INSTALLATIO,INSTALLATION BUILT IN,-2,1,0.01,59.99,0.01,66.67,66.66,99.99,Dec
535,INTERNET RADIOS,REV-ISTREA,Roberts Revival iStream 3L Black,0,2,133.00,212.49,266.00,332.49,66.49,20.00,Dec
574,RADIOS,REV-ISTREA,Roberts Revival iStream 3L Radio |,2,1,133.00,212.49,133.00,165.83,32.83,19.80,Dec
693,WASHING MACHINES,WW90DG6U8,Samsung Series 6 9kg 1400 Spin,2,1,538.00,823.99,538.00,374.17,-163.83,-43.78,Dec
699,WASHING MACHINES,WW90DG6U8,Samsung White 9kg 1400 Spin,0,1,408.84,601.99,408.84,358.33,-50.51,-14.10,Dec


In [22]:
## Compare all fields within those repeated keys
duplicate_key_summary = (
    duplicate_product_month_rows
    .groupby(
        ["Stock Code", "Source_Month"],
        dropna=False
    )
    .agg(
        Row_Count=("Stock Code", "size"),
        Description_Count=("Description", "nunique"),
        Category_Count=("Category", "nunique"),
        Sold_Period_Count=("Sold Period", "nunique"),
        Cost_Count=("Unit Cost", "nunique"),
        Price_Count=("Unit Price", "nunique"),
        Cost_Sales_Count=("Cost Sales", "nunique"),
        Sales_Value_Count=("Sales Value", "nunique"),
        Profit_Count=("Profit", "nunique"),
        Profit_Pct_Count=("Profit %", "nunique")
    )
    .reset_index()
)

display(duplicate_key_summary)

,Stock Code,Source_Month,Row_Count,Description_Count,Category_Count,Sold_Period_Count,Cost_Count,Price_Count,Cost_Sales_Count,Sales_Value_Count,Profit_Count,Profit_Pct_Count
0,INSTALLATIO,Dec,3,3,1,2,1,3,2,3,3,3
1,REV-ISTREA,Dec,2,2,2,2,1,1,2,2,2,2
2,WW90DG6U8,Dec,2,2,1,1,2,2,2,2,2,2


In [18]:
business_cols = list(master_data.columns)

exact_duplicate_mask = (
    monthly_union
    .duplicated(
        subset=business_cols,
        keep=False
    )
)

exact_duplicate_rows = (
    monthly_union[
        exact_duplicate_mask
    ]
    .sort_values(
        ["Stock Code", "Source_Month"]
    )
)

print(
    "Rows participating in exact business-row duplicates:",
    len(exact_duplicate_rows)
)

print(
    "Extra exact duplicate rows:",
    monthly_union.duplicated(
        subset=business_cols
    ).sum()
)

display(exact_duplicate_rows)

Rows participating in exact business-row duplicates: 77
Extra exact duplicate rows: 39


,Category,Stock Code,Description,Level,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Source_Month
965,WHITES ACCESSORIES,11891800,Miele Ultra Phase 2 WA UP2 1402,7,1,8.58,14.99,8.58,12.49,3.91,31.31,Jan
377,WHITES ACCESSORIES,11891800,Miele Ultra Phase 2 WA UP2 1402,7,1,8.58,14.99,8.58,12.49,3.91,31.31,Nov
455,FOOD PREP,15199,Russell Hobbs 2 Plate Portable Hob,3,1,29.99,49.49,29.99,24.99,-5.00,-20.01,Dec
79,FOOD PREP,15199,Russell Hobbs 2 Plate Portable Hob,3,1,29.99,49.49,29.99,24.99,-5.00,-20.01,Nov
459,FOOD PREP,19750,Russell Hobbs Rice Cooker 1.8Ltr,2,1,23.32,38.49,23.32,20.83,-2.49,-11.95,Dec
...,...,...,...,...,...,...,...,...,...,...,...,...
918,TUMBLE DRYERS,WTH85225IE,Bosch Series 4 8kg Heat Pump Dryer,1,1,348.30,601.49,348.30,415.83,67.53,16.24,Jan
687,WASHING MACHINES,WW90CGC04,Samsung 9kg Eco Bubble Washer,1,1,351.47,509.49,351.47,316.66,-34.81,-10.99,Dec
953,WASHING MACHINES,WW90CGC04,Samsung 9kg Eco Bubble Washer,1,1,351.47,509.49,351.47,316.66,-34.81,-10.99,Jan
572,PRINTERS,XP-2205,Epson C11CK67402 Printer,1,1,33.17,50.99,33.17,40.83,7.66,18.76,Dec


In [20]:
## Transaction behaviour of repeated product-months

display(
    duplicate_product_month_rows[
        [
            "Stock Code",
            "Description",
            "Category",
            "Source_Month",
            "Sold Period",
            "Unit Cost",
            "Unit Price",
            "Cost Sales",
            "Sales Value",
            "Profit",
            "Profit %"
        ]
    ]
)

,Stock Code,Description,Category,Source_Month,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %
521,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Dec,12,0.01,125.49,0.12,595.84,595.72,99.98
522,INSTALLATIO,INSTALLATION FREE STANDING,INSTALLATION,Dec,1,0.01,34.49,0.01,33.33,33.32,99.97
523,INSTALLATIO,INSTALLATION BUILT IN,INSTALLATION,Dec,1,0.01,59.99,0.01,66.67,66.66,99.99
535,REV-ISTREA,Roberts Revival iStream 3L Black,INTERNET RADIOS,Dec,2,133.00,212.49,266.00,332.49,66.49,20.00
574,REV-ISTREA,Roberts Revival iStream 3L Radio |,RADIOS,Dec,1,133.00,212.49,133.00,165.83,32.83,19.80
693,WW90DG6U8,Samsung Series 6 9kg 1400 Spin,WASHING MACHINES,Dec,1,538.00,823.99,538.00,374.17,-163.83,-43.78
699,WW90DG6U8,Samsung White 9kg 1400 Spin,WASHING MACHINES,Dec,1,408.84,601.99,408.84,358.33,-50.51,-14.10


In [21]:
## Reconciliation impact if duplicates were removed
dedup_test = monthly_union.drop_duplicates(
    subset=business_cols
)

impact = pd.DataFrame({
    "Metric": [
        "Rows",
        "Sold Period",
        "Cost Sales",
        "Sales Value",
        "Profit"
    ],
    "Original": [
        len(monthly_union),
        monthly_union["Sold Period"].sum(),
        monthly_union["Cost Sales"].sum(),
        monthly_union["Sales Value"].sum(),
        monthly_union["Profit"].sum()
    ],
    "After_Exact_Dedup": [
        len(dedup_test),
        dedup_test["Sold Period"].sum(),
        dedup_test["Cost Sales"].sum(),
        dedup_test["Sales Value"].sum(),
        dedup_test["Profit"].sum()
    ]
})

impact["Difference"] = (
    impact["Original"]
    - impact["After_Exact_Dedup"]
)

display(impact)

,Metric,Original,After_Exact_Dedup,Difference
0,Rows,966.00,927.00,39.00
1,Sold Period,1178.00,1138.00,40.00
2,Cost Sales,273671.79,267495.47,6176.32
3,Sales Value,279350.84,273153.50,6197.34
4,Profit,5679.05,5658.03,21.02


In [23]:
## Test true duplicates within the same month
true_duplicate_cols = [
    "Category",
    "Stock Code",
    "Description",
    "Level",
    "Sold Period",
    "Unit Cost",
    "Unit Price",
    "Cost Sales",
    "Sales Value",
    "Profit",
    "Profit %",
    "Source_Month"
]

true_duplicate_mask = monthly_union.duplicated(
    subset=true_duplicate_cols,
    keep=False
)

true_duplicate_rows = (
    monthly_union[true_duplicate_mask]
    .sort_values(
        ["Source_Month", "Stock Code"]
    )
)

print(
    "Rows participating in true same-month duplicates:",
    len(true_duplicate_rows)
)

print(
    "Extra true duplicate rows:",
    monthly_union.duplicated(
        subset=true_duplicate_cols
    ).sum()
)

display(true_duplicate_rows)

Rows participating in true same-month duplicates: 0
Extra true duplicate rows: 0


,Category,Stock Code,Description,Level,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Source_Month


In [24]:
fact_sales_stage = (
    monthly_union
    .copy()
    .reset_index(drop=True)
)

fact_sales_stage.insert(
    0,
    "Sales_Record_ID",
    range(1, len(fact_sales_stage) + 1)
)

print("Fact rows:", len(fact_sales_stage))

print(
    "Unique Sales_Record_ID:",
    fact_sales_stage["Sales_Record_ID"].nunique()
)

print(
    "Duplicate Sales_Record_ID:",
    fact_sales_stage["Sales_Record_ID"].duplicated().sum()
)

display(fact_sales_stage.head())

Fact rows: 966
Unique Sales_Record_ID: 966
Duplicate Sales_Record_ID: 0


,Sales_Record_ID,Category,Stock Code,Description,Level,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Source_Month
0,1,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,0,1,9.31,29.99,9.31,24.99,15.68,62.75,Nov
1,2,ACCESSORIES,112.204,TV Arial Lead 4.0m,3,1,1.28,2.99,1.28,2.49,1.21,48.59,Nov
2,3,ACCESSORIES,DLSC500,Delonghi Descaler,5,1,7.77,14.49,7.77,7.50,-0.27,-3.60,Nov
3,4,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,3,2,2.30,3.49,4.60,3.32,-1.28,-38.55,Nov
4,5,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),6,2,9.39,14.49,18.78,23.32,4.54,19.47,Nov


In [25]:
## Validate Sales Measures
sales_numeric_cols = [
    "Level",
    "Sold Period",
    "Unit Cost",
    "Unit Price",
    "Cost Sales",
    "Sales Value",
    "Profit",
    "Profit %"
]

numeric_quality = []

for col in sales_numeric_cols:
    numeric_quality.append({
        "Column": col,
        "Missing": fact_sales_stage[col].isna().sum(),
        "Zero": (fact_sales_stage[col] == 0).sum(),
        "Negative": (fact_sales_stage[col] < 0).sum(),
        "Min": fact_sales_stage[col].min(),
        "Max": fact_sales_stage[col].max()
    })

numeric_quality = pd.DataFrame(numeric_quality)

display(numeric_quality)

,Column,Missing,Zero,Negative,Min,Max
0,Level,0,282,72,-53.00,29.00
1,Sold Period,0,10,15,-5.00,12.00
2,Unit Cost,0,16,0,0.00,3774.00
3,Unit Price,0,0,0,0.49,5804.99
4,Cost Sales,0,24,16,-721.50,3774.00
5,Sales Value,0,0,21,-4104.19,4000.00
6,Profit,0,8,298,-4104.23,815.90
7,Profit %,0,8,281,-201.82,100.00


In [ ]:
## Validate Profit equation
fact_sales_stage["Calculated_Profit"] = (
    fact_sales_stage["Sales Value"]
    - fact_sales_stage["Cost Sales"]
)

fact_sales_stage["Profit_Difference"] = (
    fact_sales_stage["Profit"]
    - fact_sales_stage["Calculated_Profit"]
)

profit_tolerance = 0.02

profit_mismatch = fact_sales_stage[
    fact_sales_stage["Profit_Difference"].abs()
    > profit_tolerance
]

print("Rows checked          :", len(fact_sales_stage))
print("Profit matches        :", len(fact_sales_stage) - len(profit_mismatch))
print("Profit mismatches     :", len(profit_mismatch))

print(
    "Maximum absolute diff :",
    fact_sales_stage["Profit_Difference"].abs().max()
)

display(
    profit_mismatch[
        [
            "Stock Code",
            "Source_Month",
            "Cost Sales",
            "Sales Value",
            "Profit",
            "Calculated_Profit",
            "Profit_Difference"
        ]
    ].head(20)
)

Rows checked          : 966
Profit matches        : 966
Profit mismatches     : 0
Maximum absolute diff : 2.8421709430404007e-13


,Stock Code,Source_Month,Cost Sales,Sales Value,Profit,Calculated_Profit,Profit_Difference


In [27]:
## Validate Cost Sales relationship
fact_sales_stage["Calculated_Cost_Sales"] = (
    fact_sales_stage["Sold Period"]
    * fact_sales_stage["Unit Cost"]
)

fact_sales_stage["Cost_Sales_Difference"] = (
    fact_sales_stage["Cost Sales"]
    - fact_sales_stage["Calculated_Cost_Sales"]
)

cost_tolerance = 0.02

cost_mismatch = fact_sales_stage[
    fact_sales_stage["Cost_Sales_Difference"].abs()
    > cost_tolerance
]

print("Rows checked            :", len(fact_sales_stage))
print("Cost Sales matches      :", len(fact_sales_stage) - len(cost_mismatch))
print("Cost Sales mismatches   :", len(cost_mismatch))

display(
    cost_mismatch[
        [
            "Stock Code",
            "Description",
            "Source_Month",
            "Sold Period",
            "Unit Cost",
            "Cost Sales",
            "Calculated_Cost_Sales",
            "Cost_Sales_Difference"
        ]
    ].head(20)
)

Rows checked            : 966
Cost Sales matches      : 931
Cost Sales mismatches   : 35


,Stock Code,Description,Source_Month,Sold Period,Unit Cost,Cost Sales,Calculated_Cost_Sales,Cost_Sales_Difference
105,AF300UK,Ninja Dual Zone Air Fryer,Nov,0,141.66,-5.66,0.00,-5.66
118,HD301UK,Shark SppedStyle Hair Dryer,Nov,3,70.83,213.93,212.49,1.44
137,SI2641D,Smeg 60cm 7.2kW Induction Hob,Nov,1,350.57,192.94,350.57,-157.63
155,DI362DQ,Smeg Integrated Dishwasher New,Nov,2,344.06,674.80,688.12,-13.32
156,S155HVX00G,Neff N50 Integrated Dishwasher,Nov,1,432.00,450.72,432.00,18.72
220,FH90EIANT,Belling Anthracite 90cm Farmhouse,Nov,1,1102.90,1128.85,1102.90,25.95
246,SFP6301TVN,Smeg Black Single Oven,Nov,1,519.67,565.29,519.67,45.62
256,447038-01,Dyson Gen5 Detect,Nov,1,593.74,558.03,593.74,-35.71
258,470521-01,Dyson V12 Absolute Detect Slim,Nov,1,435.41,393.90,435.41,-41.51
259,476622-01,Dyson V15 Detect Total Clean,Nov,1,554.16,492.37,554.16,-61.79


In [28]:
## Validate Profit %
valid_sales_value = (
    fact_sales_stage["Sales Value"] != 0
)

fact_sales_stage["Calculated_Profit_Pct"] = np.where(
    valid_sales_value,
    (
        fact_sales_stage["Profit"]
        / fact_sales_stage["Sales Value"]
    ) * 100,
    np.nan
)

fact_sales_stage["Profit_Pct_Difference"] = (
    fact_sales_stage["Profit %"]
    - fact_sales_stage["Calculated_Profit_Pct"]
)

profit_pct_mismatch = fact_sales_stage[
    valid_sales_value
    & (
        fact_sales_stage[
            "Profit_Pct_Difference"
        ].abs() > 0.05
    )
]

print(
    "Rows with non-zero Sales Value:",
    valid_sales_value.sum()
)

print(
    "Profit % mismatches:",
    len(profit_pct_mismatch)
)

display(
    profit_pct_mismatch[
        [
            "Stock Code",
            "Source_Month",
            "Sales Value",
            "Profit",
            "Profit %",
            "Calculated_Profit_Pct",
            "Profit_Pct_Difference"
        ]
    ].head(20)
)

Rows with non-zero Sales Value: 966
Profit % mismatches: 0


,Stock Code,Source_Month,Sales Value,Profit,Profit %,Calculated_Profit_Pct,Profit_Pct_Difference


### Step 4 — Characterise sales transaction behaviour

In [ ]:
## Quantity behaviour
quantity_status = np.select(
    [
        fact_sales_stage["Sold Period"] < 0,
        fact_sales_stage["Sold Period"] == 0,
        fact_sales_stage["Sold Period"] > 0
    ],
    [
        "NEGATIVE_QUANTITY",
        "ZERO_QUANTITY",
        "POSITIVE_QUANTITY"
    ],
    default="UNKNOWN"
)

fact_sales_stage["Quantity_Status"] = quantity_status

quantity_summary = (
    fact_sales_stage
    .groupby("Quantity_Status")
    .agg(
        Records=("Sales_Record_ID", "count"),
        Units=("Sold Period", "sum"),
        Sales_Value=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
    .round(2)
)

display(quantity_summary)

,Records,Units,Sales_Value,Cost_Sales,Profit
Quantity_Status,,,,,
NEGATIVE_QUANTITY,15,-19,-3548.22,-3135.80,-412.42
POSITIVE_QUANTITY,941,1197,282951.59,276813.25,6138.34
ZERO_QUANTITY,10,0,-52.53,-5.66,-46.87


In [30]:
## Inspect negative quantities
negative_quantity_rows = (
    fact_sales_stage[
        fact_sales_stage["Sold Period"] < 0
    ]
    [
        [
            "Sales_Record_ID",
            "Category",
            "Stock Code",
            "Description",
            "Sold Period",
            "Unit Cost",
            "Unit Price",
            "Cost Sales",
            "Sales Value",
            "Profit",
            "Source_Month"
        ]
    ]
    .sort_values(["Source_Month", "Stock Code"])
)

print(
    "Negative-quantity records:",
    len(negative_quantity_rows)
)

display(negative_quantity_rows)

Negative-quantity records: 15


,Sales_Record_ID,Category,Stock Code,Description,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Source_Month
492,493,HAIRCARE,561727-01,Dyson Supersonic Nural Strawberry,-1,316.66,399.99,-270.11,-332.50,-62.39,Dec
634,635,TOASTERS,CTI4003.M,DeLonghi Distinta X 4SL Toaster,-1,51.82,84.99,-51.82,-75.00,-23.18,Dec
424,425,COOKER HOODS,DGE5861HM,AEG 80cm Canopy Cooker Hood,-1,359.55,552.99,-359.55,-340.83,18.72,Dec
416,417,COFFEE MAKERS,EC9155.MB,DeLonghi La Specailista Arte Coffee,-1,252.67,509.49,-252.67,-274.17,-21.50,Dec
558,559,KETTLES,SKE735BTR4,Sage Black Truffle Kettle,-1,67.37,109.99,-67.37,-79.17,-11.80,Dec
585,586,ROBOT CLEANING,T2080GA1,Eufy Robot Vacuum S1 Pro,-1,721.50,1274.49,-721.50,-750.00,-28.50,Dec
502,503,HEADPHONES,WHULT900NBSONY,"Black Bluetooth 5.2, NC",-1,113.51,186.99,-113.51,-99.17,14.34,Dec
877,878,SINGLE OVENS,B2ACH7AG7BNeff,N50 Graphite Grey Single Oven,-1,509.06,781.99,-509.06,-615.83,-106.77,Jan
754,755,ELECTRIC BLANKETS,DFB2004,Dimplex King Fleece Dual Ctrl,-1,53.18,84.99,-53.18,-70.79,-17.61,Jan
758,759,FOOD PREP,MQ3025,Braun 700W Handblender,-1,20.72,50.49,-20.72,-45.83,-25.11,Jan


In [31]:
## Inspect zero quantities
zero_quantity_rows = (
    fact_sales_stage[
        fact_sales_stage["Sold Period"] == 0
    ]
    [
        [
            "Sales_Record_ID",
            "Category",
            "Stock Code",
            "Description",
            "Sold Period",
            "Unit Cost",
            "Unit Price",
            "Cost Sales",
            "Sales Value",
            "Profit",
            "Source_Month"
        ]
    ]
    .sort_values(["Source_Month", "Stock Code"])
)

print(
    "Zero-quantity records:",
    len(zero_quantity_rows)
)

display(zero_quantity_rows)

Zero-quantity records: 10


,Sales_Record_ID,Category,Stock Code,Description,Sold Period,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Source_Month
669,670,TV BRACKETS,28086T,TV BRACKET 200 VESA SINGLE,0,15.29,29.99,0.00,-8.34,-8.34,Dec
481,482,FRYERS,AF400UK,Ninja Foodi Max Dual Zon Fryer 9.5L,0,157.16,254.49,0.00,0.01,0.01,Dec
655,656,TV 33 - 43,UE43U7000FKSamsung,43 in 4K Smart TV,0,231.43,360.49,0.00,-36.67,-36.67,Dec
712,713,ACCESSORIES,430009,MERCURY 4 GANG SURGE 2M,0,6.22,12.99,0.00,0.01,0.01,Jan
935,936,TV ACCESSORIES,SV1240,One For All Total Control Amplified,0,11.73,33.99,0.00,0.01,0.01,Jan
713,714,ACCESSORIES,WP100EU,Ninja Wireless Food Probe,0,66.13,95.99,0.00,-0.01,-0.01,Jan
105,106,FRYERS,AF300UK,Ninja Dual Zone Air Fryer,0,141.66,220.49,-5.66,-40.85,-35.19,Nov
60,61,DELIVERY CHARGE,Delivery,WEB DELIVERY CHARGE,0,0.00,5.99,0.00,-0.04,-0.04,Nov
117,118,HAIRCARE,HD440BPUK,SHARK FLEX STYLE – MALIBU,0,211.45,305.99,0.00,33.34,33.34,Nov
21,22,CABLES,MM524R,"Deltaco 3,5mm Male to Male Stereo",0,3.64,5.99,0.00,0.01,0.01,Nov


### Step 6 — Map Sales to Product Master and Build fact_sales

In [32]:
## Load governed Product Master
DIM_PRODUCT_FILE = PROCESSED_DIR / "dim_product.csv"

dim_product = pd.read_csv(
    DIM_PRODUCT_FILE
)

print("dim_product shape:", dim_product.shape)

display(
    dim_product[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Record_Type"
        ]
    ].head()
)

dim_product shape: (1436, 18)


,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,PRODUCT
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,PRODUCT
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,PRODUCT
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,PRODUCT
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,PRODUCT


In [33]:
## Normalize Sales Stock Code
fact_sales_stage["Product_Key"] = (
    fact_sales_stage["Stock Code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print(
    "Unique normalized Sales Product Keys:",
    fact_sales_stage["Product_Key"].nunique()
)

print(
    "Missing Product Keys:",
    fact_sales_stage["Product_Key"].isna().sum()
)

Unique normalized Sales Product Keys: 767
Missing Product Keys: 0


In [34]:
## Join to 'Product_ID'
product_map = dim_product[
    [
        "Product_ID",
        "Product_Key",
        "Record_Type"
    ]
].copy()

fact_sales_mapped = fact_sales_stage.merge(
    product_map,
    on="Product_Key",
    how="left",
    validate="many_to_one"
)

print("Source sales rows :", len(fact_sales_stage))
print("Mapped sales rows :", len(fact_sales_mapped))

print(
    "Unmapped Product_ID:",
    fact_sales_mapped["Product_ID"].isna().sum()
)


Source sales rows : 966
Mapped sales rows : 966
Unmapped Product_ID: 0


In [35]:
## Referential integrity check
mapping_validation = pd.DataFrame({
    "Check": [
        "Sales rows preserved",
        "All Product_ID mapped",
        "Unique Sales_Record_ID preserved",
        "No duplicate Sales_Record_ID"
    ],
    "Result": [
        len(fact_sales_mapped) == len(fact_sales_stage),
        fact_sales_mapped["Product_ID"].notna().all(),
        fact_sales_mapped["Sales_Record_ID"].nunique()
        == len(fact_sales_stage),
        fact_sales_mapped["Sales_Record_ID"].duplicated().sum() == 0
    ]
})

display(mapping_validation)

,Check,Result
0,Sales rows preserved,True
1,All Product_ID mapped,True
2,Unique Sales_Record_ID preserved,True
3,No duplicate Sales_Record_ID,True


In [36]:
## Add governed transaction classification
fact_sales_mapped["Transaction_Status"] = np.select(
    [
        fact_sales_mapped["Sold Period"] < 0,
        fact_sales_mapped["Sold Period"] == 0,
        fact_sales_mapped["Sold Period"] > 0
    ],
    [
        "NEGATIVE_SALES_ACTIVITY",
        "ZERO_UNIT_FINANCIAL_ADJUSTMENT",
        "POSITIVE_SALES_ACTIVITY"
    ],
    default="UNKNOWN"
)

In [37]:
## Add Cost Sales reconciliation flag
fact_sales_mapped["Calculated_Cost_Sales"] = (
    fact_sales_mapped["Sold Period"]
    * fact_sales_mapped["Unit Cost"]
)

fact_sales_mapped["Cost_Sales_Difference"] = (
    fact_sales_mapped["Cost Sales"]
    - fact_sales_mapped["Calculated_Cost_Sales"]
)

fact_sales_mapped["Cost_Sales_Reconciliation_Flag"] = np.where(
    fact_sales_mapped["Cost_Sales_Difference"].abs() <= 0.02,
    "MATCH",
    "SOURCE_ADJUSTMENT"
)

display(
    fact_sales_mapped[
        "Cost_Sales_Reconciliation_Flag"
    ]
    .value_counts()
    .to_frame("Record_Count")
)

,Record_Count
Cost_Sales_Reconciliation_Flag,
MATCH,931
SOURCE_ADJUSTMENT,35


In [38]:
## Build governed fact_sales
fact_sales = fact_sales_mapped[
    [
        "Sales_Record_ID",
        "Product_ID",
        "Product_Key",
        "Source_Month",
        "Category",
        "Stock Code",
        "Description",
        "Record_Type",
        "Level",
        "Sold Period",
        "Transaction_Status",
        "Unit Cost",
        "Unit Price",
        "Cost Sales",
        "Sales Value",
        "Profit",
        "Profit %",
        "Cost_Sales_Reconciliation_Flag"
    ]
].copy()

fact_sales = fact_sales.sort_values(
    "Sales_Record_ID"
).reset_index(drop=True)

print("fact_sales shape:", fact_sales.shape)

display(fact_sales.head(10))

fact_sales shape: (966, 18)


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
0,1,1223,TLS169BOXE,Nov,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,9.31,29.99,9.31,24.99,15.68,62.75,MATCH
1,2,17,112.204,Nov,ACCESSORIES,112.204,TV Arial Lead 4.0m,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,1.28,2.99,1.28,2.49,1.21,48.59,MATCH
2,3,385,DLSC500,Nov,ACCESSORIES,DLSC500,Delonghi Descaler,PRODUCT,5,1,POSITIVE_SALES_ACTIVITY,7.77,14.49,7.77,7.50,-0.27,-3.60,MATCH
3,4,246,AF01,Nov,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,PRODUCT,3,2,POSITIVE_SALES_ACTIVITY,2.30,3.49,4.60,3.32,-1.28,-38.55,MATCH
4,5,1076,SES007NEU0,Nov,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),PRODUCT,6,2,POSITIVE_SALES_ACTIVITY,9.39,14.49,18.78,23.32,4.54,19.47,MATCH
5,6,1074,SEC250NEU0,Nov,ACCESSORIES,SEC250NEU0,Sage Espresso Cleaning Tablets,PRODUCT,0,2,POSITIVE_SALES_ACTIVITY,9.39,16.99,18.78,23.32,4.54,19.47,MATCH
6,7,1077,SES008WHT0SAGE,Nov,ACCESSORIES,SES008WHT0Sage,Claro Swiss Water Filter,PRODUCT,11,3,POSITIVE_SALES_ACTIVITY,9.39,14.49,28.17,35.01,6.84,19.54,MATCH
7,8,1162,SV93605G,Nov,ACCESSORIES,SV93605G,OFA Digi ANT Indoor Amp to 45dB,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,19.17,33.99,19.17,28.33,9.16,32.33,MATCH
8,9,526,GO524,Nov,ACCESSORIES,GO524,GO IRE to EU Travel Adaptor,PRODUCT,1,1,POSITIVE_SALES_ACTIVITY,4.96,9.49,4.96,7.49,2.53,33.78,MATCH
9,10,120,430009,Nov,ACCESSORIES,430009,MERCURY 4 GANG SURGE 2M,PRODUCT,2,1,POSITIVE_SALES_ACTIVITY,6.22,12.99,6.22,9.17,2.95,32.17,MATCH


In [39]:
## Final financial reconciliation
final_reconciliation = pd.DataFrame({
    "Metric": [
        "Rows",
        "Sold Period",
        "Cost Sales",
        "Sales Value",
        "Profit"
    ],
    "Source_Master": [
        len(master_data),
        master_data["Sold Period"].sum(),
        master_data["Cost Sales"].sum(),
        master_data["Sales Value"].sum(),
        master_data["Profit"].sum()
    ],
    "Governed_Fact": [
        len(fact_sales),
        fact_sales["Sold Period"].sum(),
        fact_sales["Cost Sales"].sum(),
        fact_sales["Sales Value"].sum(),
        fact_sales["Profit"].sum()
    ]
})

final_reconciliation["Difference"] = (
    final_reconciliation["Governed_Fact"]
    - final_reconciliation["Source_Master"]
)

display(final_reconciliation)

,Metric,Source_Master,Governed_Fact,Difference
0,Rows,966.00,966.00,0.0
1,Sold Period,1178.00,1178.00,0.0
2,Cost Sales,273671.79,273671.79,0.0
3,Sales Value,279350.84,279350.84,0.0
4,Profit,5679.05,5679.05,0.0


In [40]:
## Product and transaction summary
print(
    "Unique Product_ID:",
    fact_sales["Product_ID"].nunique()
)

print("\nRecord types:")
display(
    fact_sales["Record_Type"]
    .value_counts()
    .to_frame("Records")
)

print("\nTransaction status:")
display(
    fact_sales["Transaction_Status"]
    .value_counts()
    .to_frame("Records")
)

Unique Product_ID: 767

Record types:


,Records
Record_Type,
PRODUCT,960
SERVICE_CHARGE,6



Transaction status:


,Records
Transaction_Status,
POSITIVE_SALES_ACTIVITY,941
NEGATIVE_SALES_ACTIVITY,15
ZERO_UNIT_FINANCIAL_ADJUSTMENT,10


In [42]:
financial_tolerance = 0.01

sales_fact_gate = (
    len(fact_sales) == 966
    and fact_sales["Sales_Record_ID"].is_unique
    and fact_sales["Product_ID"].notna().all()
    and fact_sales["Product_Key"].notna().all()
    and fact_sales["Sold Period"].sum()
        == master_data["Sold Period"].sum()
    and abs(
        fact_sales["Cost Sales"].sum()
        - master_data["Cost Sales"].sum()
    ) <= financial_tolerance
    and abs(
        fact_sales["Sales Value"].sum()
        - master_data["Sales Value"].sum()
    ) <= financial_tolerance
    and abs(
        fact_sales["Profit"].sum()
        - master_data["Profit"].sum()
    ) <= financial_tolerance
)

print(
    " SALES FACT GATE:",
    "PASSED" if sales_fact_gate else "FAILED"
)

 SALES FACT GATE: PASSED


In [ ]:
## Save the fact_sales

""" FACT_SALES_FILE = PROCESSED_DIR / "fact_sales.csv"

fact_sales.to_csv(
    FACT_SALES_FILE,
    index=False
)

print("Saved:", FACT_SALES_FILE) """

Saved: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\fact_sales.csv


In [44]:
fact_sales_check = pd.read_csv(
    FACT_SALES_FILE
)

print("Saved rows:", len(fact_sales_check))
print(
    "Unique Sales_Record_ID:",
    fact_sales_check["Sales_Record_ID"].nunique()
)
print(
    "Missing Product_ID:",
    fact_sales_check["Product_ID"].isna().sum()
)
print(
    "Sold Period total:",
    fact_sales_check["Sold Period"].sum()
)
print(
    "Sales Value total:",
    fact_sales_check["Sales Value"].sum()
)
print(
    "Profit total:",
    fact_sales_check["Profit"].sum()
)

Saved rows: 966
Unique Sales_Record_ID: 966
Missing Product_ID: 0
Sold Period total: 1178
Sales Value total: 279350.83999999997
Profit total: 5679.049999999999


### Step 7 — Profile Level

In [45]:
## Basic Level profile
level_profile = pd.Series({
    "Total_Rows": len(fact_sales),
    "Missing_Level": fact_sales["Level"].isna().sum(),
    "Negative_Level": (fact_sales["Level"] < 0).sum(),
    "Zero_Level": (fact_sales["Level"] == 0).sum(),
    "Positive_Level": (fact_sales["Level"] > 0).sum(),
    "Minimum_Level": fact_sales["Level"].min(),
    "Maximum_Level": fact_sales["Level"].max(),
    "Mean_Level": fact_sales["Level"].mean(),
    "Median_Level": fact_sales["Level"].median()
})

display(level_profile.to_frame("Value"))

,Value
Total_Rows,966.00000
Missing_Level,0.00000
Negative_Level,72.00000
Zero_Level,282.00000
Positive_Level,612.00000
Minimum_Level,-53.00000
Maximum_Level,29.00000
Mean_Level,1.07971
Median_Level,1.00000


In [46]:
## Inspect negative Level records
negative_level_rows = (
    fact_sales[
        fact_sales["Level"] < 0
    ]
    [
        [
            "Sales_Record_ID",
            "Product_ID",
            "Category",
            "Stock Code",
            "Description",
            "Source_Month",
            "Level",
            "Sold Period",
            "Sales Value",
            "Profit",
            "Transaction_Status"
        ]
    ]
    .sort_values(
        ["Level", "Source_Month"]
    )
)

print(
    "Negative Level records:",
    len(negative_level_rows)
)

display(negative_level_rows)

Negative Level records: 72


,Sales_Record_ID,Product_ID,Category,Stock Code,Description,Source_Month,Level,Sold Period,Sales Value,Profit,Transaction_Status
521,522,615,INSTALLATION,INSTALLATIO,INSTALLATION FEE,Dec,-53,12,595.84,595.72,POSITIVE_SALES_ACTIVITY
809,810,615,INSTALLATION,INSTALLATIO,INSTALLATION FEE,Jan,-53,8,408.35,408.27,POSITIVE_SALES_ACTIVITY
150,151,615,INSTALLATION,INSTALLATIO,INSTALLATION FEE,Nov,-53,7,308.33,308.26,POSITIVE_SALES_ACTIVITY
439,440,369,DELIVERY CHARGE,Delivery,WEB DELIVERY CHARGE,Dec,-26,5,10.01,10.01,POSITIVE_SALES_ACTIVITY
748,749,369,DELIVERY CHARGE,Delivery,WEB DELIVERY CHARGE,Jan,-26,4,13.33,13.33,POSITIVE_SALES_ACTIVITY
...,...,...,...,...,...,...,...,...,...,...,...
256,257,137,STICK VACS,447038-01,Dyson Gen5 Detect,Nov,-1,1,495.83,-62.20,POSITIVE_SALES_ACTIVITY
292,293,1390,TUMBLE DRYERS,WQG245R1G,Bosch Graphite 9kg Heat Pump Dryer,Nov,-1,2,1215.00,114.20,POSITIVE_SALES_ACTIVITY
297,298,121,TV 33 - 43,43LQ60006LA.LG,"43"" Smart TV",Nov,-1,5,795.82,-139.18,POSITIVE_SALES_ACTIVITY
341,342,1008,USA F/F,RF24BB620ESSamsung,French Dr St/St,Nov,-1,1,1099.17,-359.06,POSITIVE_SALES_ACTIVITY


In [47]:
## Relationship between Level and Sold Period
level_quantity_relationship = pd.DataFrame({
    "Metric": [
        "Level equals Sold Period",
        "Level differs from Sold Period",
        "Negative Level + Positive Sold Period",
        "Negative Level + Zero Sold Period",
        "Negative Level + Negative Sold Period"
    ],
    "Records": [
        (fact_sales["Level"] == fact_sales["Sold Period"]).sum(),
        (fact_sales["Level"] != fact_sales["Sold Period"]).sum(),

        (
            (fact_sales["Level"] < 0)
            & (fact_sales["Sold Period"] > 0)
        ).sum(),

        (
            (fact_sales["Level"] < 0)
            & (fact_sales["Sold Period"] == 0)
        ).sum(),

        (
            (fact_sales["Level"] < 0)
            & (fact_sales["Sold Period"] < 0)
        ).sum()
    ]
})

display(level_quantity_relationship)

,Metric,Records
0,Level equals Sold Period,257
1,Level differs from Sold Period,709
2,Negative Level + Positive Sold Period,70
3,Negative Level + Zero Sold Period,1
4,Negative Level + Negative Sold Period,1


In [48]:
## Negative Level by month
negative_level_month = (
    fact_sales[
        fact_sales["Level"] < 0
    ]
    .groupby("Source_Month")
    .agg(
        Records=("Sales_Record_ID", "count"),
        Products=("Product_ID", "nunique"),
        Minimum_Level=("Level", "min"),
        Total_Level=("Level", "sum"),
        Units_Sold=("Sold Period", "sum"),
        Sales_Value=("Sales Value", "sum"),
        Profit=("Profit", "sum")
    )
    .round(2)
)

display(negative_level_month)

,Records,Products,Minimum_Level,Total_Level,Units_Sold,Sales_Value,Profit
Source_Month,,,,,,,
Dec,29,26,-53,-162,63,6628.64,809.05
Jan,20,19,-53,-144,37,-1811.74,-3635.41
Nov,23,22,-53,-130,37,6481.99,-299.15


In [49]:
## Negative Level by category
negative_level_category = (
    fact_sales[
        fact_sales["Level"] < 0
    ]
    .groupby("Category")
    .agg(
        Records=("Sales_Record_ID", "count"),
        Products=("Product_ID", "nunique"),
        Minimum_Level=("Level", "min"),
        Total_Level=("Level", "sum"),
        Units_Sold=("Sold Period", "sum"),
        Sales_Value=("Sales Value", "sum")
    )
    .sort_values(
        "Records",
        ascending=False
    )
    .round(2)
)

display(negative_level_category)

,Records,Products,Minimum_Level,Total_Level,Units_Sold,Sales_Value
Category,,,,,,
DELIVERY CHARGE,9,2,-26,-147,22,164.97
INSTALLATION,5,1,-53,-164,29,1412.52
FOOD PREP,4,3,-7,-16,9,508.30
SECURITY,4,2,-2,-6,6,374.14
TV 33 - 43,4,2,-1,-4,9,1420.82
TV 51 - 59,4,2,-2,-7,6,2861.67
COFFEE ACCESSORIES,3,1,-1,-3,4,92.44
INK,3,2,-1,-3,4,56.64
HOME CINEMA,3,3,-2,-4,3,590.82


In [50]:
## Most extreme negative Levels
display(
    negative_level_rows[
        [
            "Stock Code",
            "Description",
            "Category",
            "Source_Month",
            "Level",
            "Sold Period",
            "Sales Value",
            "Transaction_Status"
        ]
    ]
    .head(20)
)

,Stock Code,Description,Category,Source_Month,Level,Sold Period,Sales Value,Transaction_Status
521,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Dec,-53,12,595.84,POSITIVE_SALES_ACTIVITY
809,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Jan,-53,8,408.35,POSITIVE_SALES_ACTIVITY
150,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Nov,-53,7,308.33,POSITIVE_SALES_ACTIVITY
439,Delivery,WEB DELIVERY CHARGE,DELIVERY CHARGE,Dec,-26,5,10.01,POSITIVE_SALES_ACTIVITY
748,Delivery,WEB DELIVERY CHARGE,DELIVERY CHARGE,Jan,-26,4,13.33,POSITIVE_SALES_ACTIVITY
60,Delivery,WEB DELIVERY CHARGE,DELIVERY CHARGE,Nov,-26,0,-0.04,ZERO_UNIT_FINANCIAL_ADJUSTMENT
570,M,MISCELANEOUS,MISC,Dec,-22,8,150.00,POSITIVE_SALES_ACTIVITY
856,M,MISCELANEOUS,MISC,Jan,-22,4,-4104.19,POSITIVE_SALES_ACTIVITY
438,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,Dec,-14,3,34.17,POSITIVE_SALES_ACTIVITY
747,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,Jan,-14,3,3.33,POSITIVE_SALES_ACTIVITY


##  Conclusion

- `Level` represents stock quantity present/on hand at the reporting point.
- `Sold Period` represents quantity sold during the reporting period.
- `Level` and `Sold Period` are therefore independent measures and should not be expected to match.
- Negative `Level` values for physical products represent negative stock positions and are preserved as valid source-system values.
- Service records such as Installation and Delivery are non-physical items; therefore, their `Level` values must not be interpreted as physical inventory.
- Service records will be excluded from inventory availability, stockout, reorder and inventory-optimisation KPIs.
- No `Level` values are modified or removed during Phase 2.

**Decision:** Preserve the source `Level` field unchanged and apply inventory interpretation only to physical products.